In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import urllib.parse


# -----------------------------
# PRODUCT DETAIL EXTRACTION
# -----------------------------
import re

def clean_url(url):
    matches = re.findall(r'https://www\.amazon\.in/[^\s]+', url)
    return matches[0] if matches else None

def get_title(soup):
    try:
        title = soup.find("span", attrs={"id": "productTitle"}).text.strip()
    except:
        title = ""
    return title


def get_price(soup):
    try:
        price = soup.find("span", attrs={"class": "a-price-whole"}).text.strip()
    except:
        price = ""
    return price


def get_rating(soup):
    try:
        rating = soup.find("span", attrs={"class": "a-icon-alt"}).text.strip()
    except:
        rating = ""
    return rating


def get_review_count(soup):
    try:
        reviews = soup.find("span", attrs={"id": "acrCustomerReviewText"}).text.strip()
    except:
        reviews = ""
    return reviews


def get_availability(soup):
    try:
        available = soup.find("div", attrs={"id": "availability"}).find("span").text.strip()
    except:
        available = "Not Available"
    return available


# -----------------------------
# MAIN SCRAPER
# -----------------------------
if __name__ == "__main__":

    URL = "https://www.amazon.in/s?k=phones"

    HEADERS = ({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36',
        'Accept-Language': 'en-US,en;q=0.5'
    })

    # Request search result page
    webpage = requests.get(URL, headers=HEADERS)
    soup = BeautifulSoup(webpage.content, "html.parser")

    # Fetch product links
    links = soup.find_all("a", attrs={
        "class": "a-link-normal s-no-outline"
    })

    links_list = []

    for link in links:
        raw_href = link.get("href")

        if raw_href is None:
            continue

        # Handle Amazon tracking URLs
        if "url=" in raw_href:
            try:
                encoded = raw_href.split("url=")[1].split("&")[0]
                decoded = urllib.parse.unquote(encoded)
                links_list.append(decoded)
            except:
                continue
        else:
            links_list.append(raw_href)

    print(f"Total cleaned links: {len(links_list)}")

    d = {"title": [], "price": [], "rating": [], "reviews": [], "availability": []}

    # Loop through product links and extract data
    for link in links_list:

        # Use correct domain
        product_url = "https://www.amazon.in" + link

        try:
            new_webpage = requests.get(product_url, headers=HEADERS)
            new_soup = BeautifulSoup(new_webpage.content, "html.parser")

            d['title'].append(get_title(new_soup))
            d['price'].append(get_price(new_soup))
            d['rating'].append(get_rating(new_soup))
            d['reviews'].append(get_review_count(new_soup))
            d['availability'].append(get_availability(new_soup))

        except Exception as e:
            print("Error processing:", product_url)
            print(e)
            continue

    # Create dataframe
    amazon_df = pd.DataFrame.from_dict(d)

    # Clean missing titles
    amazon_df = amazon_df.replace({'title': {'': np.nan}})
    amazon_df = amazon_df.dropna(subset=['title'])

    # Export CSV
    amazon_df.to_csv("amazon_data.csv", index=False)
    print("Scraping completed. File saved as amazon_data.csv")


Total cleaned links: 20
Error processing: https://www.amazon.inhttps://aax-eu-zaz.amazon.in/x/c/JNIzoRJ10eaZoOmHVVXTSMAAAAGal72LaQoAAAH2AQBvbm9fdHhuX2JpZDIgICBvbm9fdHhuX2ltcDEgICCNhBV0/clv1_CEuOPUxokZA0iHrVVPsxhRn2TSlUWLRu7c0C3CeWyvp7x39t6mmNPP018HHz9hvVjq0yzpNb3X7OfU0BBWcAX0pENwyXJvCuC0QCOHB9dWWtOqdjlEfB8IkGXCQDVeW6XJeXalC7rdIQEQQ8kV_3ZVhXLA8QC8x8H15tWbp4Xgyay_2WjkwrSxutOI2vKbYuDzUcfMyagcpNBHhtJZhsefQQh8yliMIIKin6O61_zAYOsoJhkeWuV2gvjBmW5lJBN2PMN-SRnR9AjX0lL9xFahCwdXiTzSjwM7ue5YyzaIsMZGWl1aK1xkuNcA7FBIOT3_cq0eVrbNPflr72FsG9JPtq6s9XBzId9qg_3tBqLp5u61N7ChKd1SWr3JYqfnqDdhxLjmOcuEwt_RDIKaKtZk4Af6rZhPBne-wSBml9tlZkUux_rAVrRF9I0bxcInIFH-Gzc_r6mO-bWyVNaReipMUTeUmOk2KifChiJ4snkeH1E39g8pKg8xPmTmFjVYOMr_VkwTthKoxWmj2Ajwi85x_bWgR-Y2A4Ly61XW_0GbfWKhOMGTd0mEAZAg3OfkfrU_0Pj89oQgyJhpZTgsU3XVSZOi98lgp6zP47jZipuW8Vvb-f_ZYgVnHu5zjScpWFM2scS8o1ABhn481pGcx6qRBCz6fwPTJDdSAwuu949KUrOXwgQT2Fh0Uahx_7TcN04wYF6H5r9toxosoiE2-APmtF2u7jMd_d-6fxImhUPS323DbfddJV7B9Gz_QIVemtp_tcMhLTn2TR5RPMYAbeH3RllibLAkpWt8pDeNEBQS

In [11]:
amazon_df

,title,price,rating,reviews,availability
0,OnePlus Nord 5 | Snapdragon 8s Gen 3 | Stable ...,"31,998.",4.4 out of 5 stars,"3,120 ratings",In stock
1,"Samsung Galaxy M17 5G (Sapphire Black, 8GB RAM...","16,999.",4.2 out of 5 stars,248 ratings,In stock
2,"POCO C71, Desert Gold (6GB, 128GB)","6,799.",4.1 out of 5 stars,"1,169 ratings",In stock
3,"Redmi A4 5G (Sparkle Purple, 4GB RAM, 128GB St...","8,599.",4.0 out of 5 stars,"9,297 ratings",In stock
4,Nothing Phone (3a) Black 128GB 8GB RAM,"23,200.",4.4 out of 5 stars,390 ratings,Only 1 left in stock.
5,"realme NARZO 80 Lite 4G (Obsidian Black, 6GB+1...","8,298.",4.2 out of 5 stars,"1,539 ratings",In stock
6,Samsung Galaxy S24 FE 5G AI Smartphone (Graphi...,"31,999.",4.3 out of 5 stars,"1,137 ratings",
7,OnePlus Nord CE5 | Massive 7100mAh Battery | M...,"24,998.",4.4 out of 5 stars,"3,138 ratings",In stock
8,"Motorola G96 (Pastures, 8GB RAM, 128GB Storage)","16,345.",4.4 out of 5 stars,82 ratings,In stock
9,"OnePlus Nord CE4 (Celadon Marble, 8GB RAM, 128...","18,999.",4.3 out of 5 stars,"11,994 ratings",In stock
